# 🧠 Module 00: The Genesis & The Atomic Neuron (1943 - 1969)
### *From Biological Inspiration to the Perceptron and the XOR Crisis*

---

## 📜 Historical Prelude: Where It All Began
In 1943, neurophysiologist **Warren McCulloch** and 18-year-old logician **Walter Pitts** asked a radical question:
> *"Can the electrochemical firing of biological brain cells perform formal logic?"*

They designed the first mathematical model of a neuron. Fourteen years later in 1957, **Frank Rosenblatt** took the next historic leap: **The Perceptron**, a machine that could learn its own weights from trial and error.

In this notebook, you will:
1. **Build the 1943 McCulloch-Pitts Neuron** and construct biological logic gates (AND, OR, NOT).
2. **Implement Rosenblatt's 1957 Perceptron** with the Perceptron Learning Rule from raw mathematical scratch.
3. **Visualize 2D Decision Boundaries** dynamically using Matplotlib.
4. **Recreate the 1969 XOR Crisis** that caused the First AI Winter.
5. **Conquer XOR with a Multilayer Perceptron (MLP)** in PyTorch, witnessing how hidden representations warp space.

## 1. The 1943 McCulloch-Pitts Neuron

### The Biological Metaphor:
- **Dendrites**: Receive inputs $x_1, x_2, \dots, x_n \in \{0, 1\}$.
- **Excitatory signals**: Push the neuron toward firing (+1 charge each).
- **Inhibitory signals**: Instant emergency brake! If active, the neuron *cannot* fire.
- **Threshold $\theta$**: If the sum of excitatory inputs reaches $\theta$, the neuron spikes an action potential ($y=1$).

In [ ]:
class McCullochPittsNeuron:
    def __init__(self, threshold: int, num_excitatory: int):
        self.threshold = threshold
        self.weights = [1] * num_excitatory
        
    def forward(self, excitatory: list, inhibitory: list = None) -> int:
        if inhibitory and any(inh == 1 for inh in inhibitory):
            return 0  # Absolute veto
        total_excitation = sum(w * x for w, x in zip(self.weights, excitatory))
        return 1 if total_excitation >= self.threshold else 0

# Building AND, OR, and NOT gates
and_neuron = McCullochPittsNeuron(threshold=2, num_excitatory=2)
or_neuron = McCullochPittsNeuron(threshold=1, num_excitatory=2)

print("AND Gate Truth Table:")
for x1, x2 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    print(f"  {x1} AND {x2} = {and_neuron.forward([x1, x2])}")

print("\nOR Gate Truth Table:")
for x1, x2 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    print(f"  {x1} OR  {x2} = {or_neuron.forward([x1, x2])}")

## 2. The 1957 Rosenblatt Perceptron

The limitation of McCulloch-Pitts was that its weights were fixed by hand. Frank Rosenblatt invented a rule for the neuron to **learn** from mistakes:

$$\hat{y} = \begin{cases} 1 & \text{if } \mathbf{w} \cdot \mathbf{x} + b \ge 0 \\ 0 & \text{if } \mathbf{w} \cdot \mathbf{x} + b < 0 \end{cases}$$

### The Perceptron Learning Rule:
$$\Delta \mathbf{w} = \eta \cdot (y - \hat{y}) \cdot \mathbf{x}$$
$$\Delta b = \eta \cdot (y - \hat{y})$$

Where $\eta$ is the learning rate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class RosenblattPerceptron:
    def __init__(self, n_inputs: int, lr: float = 0.1, max_epochs: int = 50):
        self.lr = lr
        self.max_epochs = max_epochs
        self.weights = np.zeros(n_inputs)
        self.bias = 0.0
        self.history = []
        
    def predict(self, x: np.ndarray) -> int:
        return 1 if (np.dot(x, self.weights) + self.bias) >= 0.0 else 0
        
    def fit(self, X: np.ndarray, y: np.ndarray):
        for epoch in range(self.max_epochs):
            errors = 0
            for xi, target in zip(X, y):
                y_hat = self.predict(xi)
                err = target - y_hat
                if err != 0:
                    errors += 1
                    self.weights += self.lr * err * xi
                    self.bias += self.lr * err
            self.history.append((self.weights.copy(), self.bias))
            if errors == 0:
                print(f"Converged in epoch {epoch + 1}!")
                break
        return self

# Train on OR gate
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([0, 1, 1, 1])

p = RosenblattPerceptron(n_inputs=2, lr=0.2)
p.fit(X, y)
print(f"Final Weights: {p.weights}, Bias: {p.bias:.2f}")

### Visualizing the 2D Decision Boundary
The decision boundary occurs where $\mathbf{w} \cdot \mathbf{x} + b = 0$, which gives the line:
$$x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$$

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
colors = ['red' if label == 0 else 'blue' for label in y]
ax.scatter(X[:, 0], X[:, 1], c=colors, s=150, edgecolors='black', zorder=5)

# Plot decision boundary
x_vals = np.linspace(-0.5, 1.5, 100)
w1, w2 = p.weights
b = p.bias
y_vals = -(w1 * x_vals + b) / w2

ax.plot(x_vals, y_vals, 'k--', label=f'Decision Boundary: {w1:.1f}x1 + {w2:.1f}x2 + {b:.1f} = 0')
ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.set_xlabel('Input x1', fontsize=12)
ax.set_ylabel('Input x2', fontsize=12)
ax.set_title('Rosenblatt Perceptron: OR Gate Decision Boundary', fontsize=14)
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend()
plt.show()

## 3. The 1969 XOR Crisis & The AI Winter

Let us now attempt to train on XOR: $(0,0)\to 0, (0,1)\to 1, (1,0)\to 1, (1,1)\to 0$.
Observe how the single perceptron fails to separate the diagonal points!

In [ ]:
y_xor = np.array([0, 1, 1, 0])
p_xor = RosenblattPerceptron(n_inputs=2, lr=0.1, max_epochs=30)
p_xor.fit(X, y_xor)

preds = [p_xor.predict(xi) for xi in X]
print("XOR Predictions:", preds, "Expected:", y_xor.tolist())

# Visualizing why XOR is linearly non-separable
fig, ax = plt.subplots(figsize=(6, 6))
for i, (pt, label) in enumerate(zip(X, y_xor)):
    c = 'blue' if label == 1 else 'red'
    ax.scatter(pt[0], pt[1], color=c, s=200, edgecolors='black', zorder=5)
    ax.annotate(f"({int(pt[0])},{int(pt[1])}) -> {label}", (pt[0]+0.05, pt[1]+0.05), fontsize=11)

ax.plot([0, 1], [1, 0], 'g:', label='Class 1 Diagonal')
ax.plot([0, 1], [0, 1], 'm:', label='Class 0 Diagonal')
ax.set_xlim(-0.2, 1.4)
ax.set_ylim(-0.2, 1.4)
ax.set_title("The XOR Non-Separable Dilemma (1969)", fontsize=14)
ax.legend()
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()

## 4. Modern Resolution: PyTorch Multilayer Perceptron (MLP)

By adding a hidden layer with a non-linear activation (ReLU), the network learns to warp the coordinate space, making XOR linearly separable!

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class XOR_MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)
        self.act = nn.ReLU()
        self.output = nn.Linear(4, 1)
        
    def forward(self, x):
        return torch.sigmoid(self.output(self.act(self.hidden(x))))

torch.manual_seed(42)
model = XOR_MLP()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.08)

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y_xor, dtype=torch.float32).unsqueeze(1)

for epoch in range(300):
    optimizer.zero_grad()
    loss = criterion(model(X_t), y_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    final_probs = model(X_t)
    print("Trained PyTorch MLP Outputs on XOR:")
    for i in range(4):
        print(f"  Input {X[i].tolist()} -> Prob: {final_probs[i].item():.4f} (Target: {y_xor[i]})")